# Lab de Limpeza - Camada Silver: Clima

**Comentário:** Descreva aqui as regras de negócio identificadas durante a limpeza manual.

In [9]:
import pandas as pd
import s3fs

In [10]:
storage_options = {
    "key": "minio",
    "secret": "minio123",
    "client_kwargs": {"endpoint_url": "http://localhost:9000"}
}

In [11]:
fs = s3fs.S3FileSystem(**storage_options)
arquivos = fs.glob("bronze/clima/**/*.csv")
arquivos

['bronze/clima/ingestion_date=2026-04-29T00:25:03/clima.csv']

In [12]:
df = pd.concat(
    [
        pd.read_csv(f"s3://{arquivo}", storage_options=storage_options) 
        for arquivo in arquivos
    ],
    ignore_index=True
)
df.head()

,time,temperature_2m_max,precipitation_sum,_ingested_at,_api_source
0,2024-01-01,21.1,9.6,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive
1,2024-01-02,24.8,1.9,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive
2,2024-01-03,24.9,10.2,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive
3,2024-01-04,25.5,16.2,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive
4,2024-01-05,26.5,1.5,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive


In [13]:
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df["_ingested_at"] = pd.to_datetime(df["_ingested_at"], errors="coerce")
df["temperature_2m_max"] = pd.to_numeric(df["temperature_2m_max"], errors="coerce")
df["precipitation_sum"] = pd.to_numeric(df["precipitation_sum"], errors="coerce")

In [14]:
df = df.sort_values("time")

In [15]:
df = df.sort_values("_ingested_at")
df = df.drop_duplicates(subset=["time"], keep="last")

In [16]:
df = df.sort_values("time")
df

,time,temperature_2m_max,precipitation_sum,_ingested_at,_api_source
0,2024-01-01,21.1,9.6,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive
1,2024-01-02,24.8,1.9,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive
2,2024-01-03,24.9,10.2,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive
3,2024-01-04,25.5,16.2,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive
4,2024-01-05,26.5,1.5,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive
...,...,...,...,...,...
361,2024-12-27,24.8,11.1,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive
362,2024-12-28,26.4,12.8,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive
363,2024-12-29,24.2,19.8,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive
364,2024-12-30,25.9,7.2,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive


In [17]:
df[["temperature_2m_max", "precipitation_sum"]].describe()

,temperature_2m_max,precipitation_sum
count,366.000000,366.000000
mean,25.983333,3.480601
std,3.088959,6.465184
min,12.700000,0.000000
25%,24.000000,0.000000
50%,26.100000,0.300000
75%,28.350000,3.575000
max,32.600000,41.600000


In [18]:
# 	temperature_2m_max -> Maximum  daily air temperature at 2 meters above ground
# 	precipitation_sum ->Sum of daily precipitation (including rain, showers and snowfall)

Temperatura (temperature_2m_max)

Natureza
- variável contínua
- distribuição quase normal
- limites físicos conhecidos

IQR (baseado no dado)

In [19]:
Q1 = df["temperature_2m_max"].quantile(0.25)
Q3 = df["temperature_2m_max"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df["temp_outlier_iqr"] = ~df["temperature_2m_max"].between(lower, upper)

Limite físico (sanity check)

In [20]:
df["temp_outlier_fisico"] = ~df["temperature_2m_max"].between(-10, 50)

In [21]:
df["temp_outlier"] = df["temp_outlier_iqr"] | df["temp_outlier_fisico"]

Precipitação (precipitation_sum)

Natureza
- soma diária
- zero-inflated
- altamente assimétrica
- eventos extremos são reais

Percentil Alto

In [22]:
p99 = df["precipitation_sum"].quantile(0.99)

df["precip_extremo"] = df["precipitation_sum"] > p99

Limite físico mínimo

In [23]:
df["precip_invalido"] = df["precipitation_sum"] < 0

In [24]:
df["precip_outlier"] = df["precip_extremo"] | df["precip_invalido"]

In [26]:
df['time'] = pd.to_datetime(df['time'])

In [27]:
df['ano'] = df['time'].dt.year
df['mes'] = df['time'].dt.month

In [28]:
df.head()

,time,temperature_2m_max,precipitation_sum,_ingested_at,_api_source,temp_outlier_iqr,temp_outlier_fisico,temp_outlier,precip_extremo,precip_invalido,precip_outlier,ano,mes
0,2024-01-01,21.1,9.6,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive,False,False,False,False,False,False,2024,1
1,2024-01-02,24.8,1.9,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive,False,False,False,False,False,False,2024,1
2,2024-01-03,24.9,10.2,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive,False,False,False,False,False,False,2024,1
3,2024-01-04,25.5,16.2,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive,False,False,False,False,False,False,2024,1
4,2024-01-05,26.5,1.5,2026-04-29 00:25:03.420132,https://archive-api.open-meteo.com/v1/archive,False,False,False,False,False,False,2024,1


In [30]:
df_silver = df[["time", "temperature_2m_max", "precipitation_sum", "_ingested_at", "_api_source", "ano", "mes"]]

In [ ]:
try:
    df_silver.to_parquet(
        path="s3://silver/clima/",
        partition_cols=["ano", "mes"],
        storage_options=storage_options,
        index=False,
        compression="snappy"
    )
    print("✓ Dados salvos com sucesso na camada Silver (MinIO)!")
except Exception as e:
    print(f"✗ Erro ao salvar na camada Silver: {e}")

✓ Dados salvos com sucesso na camada Silver (MinIO)!
